In [1]:
import os
import pandas as pd

files = sorted(os.listdir("data/raw"))
frames = [pd.read_csv("data/raw/" + f) for f in files]
df = pd.concat(frames, ignore_index=True)
df.shape

(2398, 22)

In [2]:
totals = df[df["Org Code"] == "TOTAL"]
df = df[df["Org Code"] != "TOTAL"].copy()

att_cols = [c for c in df.columns if c.startswith("A&E attendances")]
over_cols = [c for c in df.columns if c.startswith("Attendances over 4hrs")]

print("Our sum:   ", df[att_cols].sum().sum())
print("NHS total: ", totals[att_cols].sum().sum())
print("Rows left: ", df.shape)

Our sum:    27976025
NHS total:  27976025
Rows left:  (2386, 22)


In [3]:
df["Parent Org"] = df["Parent Org"].str.strip()
df["month"] = pd.to_datetime(df["Period"].str.replace("MSitAE-", ""), format="%B-%Y")
df["month"].value_counts().sort_index()

month
2025-04-01    198
2025-05-01    200
2025-06-01    201
2025-07-01    201
2025-08-01    201
2025-09-01    201
2025-10-01    197
2025-11-01    197
2025-12-01    197
2026-01-01    197
2026-02-01    197
2026-03-01    199
Name: count, dtype: int64

In [4]:
df["total_attendances"] = df[att_cols].sum(axis=1)
df["over_4hrs"] = df[over_cols].sum(axis=1)
df["pct_within_4hrs"] = ((1 - df["over_4hrs"] / df["total_attendances"]) * 100).round(1)
df = df.rename(columns={"Patients who have waited 12+ hrs from DTA to admission": "waits_12hrs"})
df["has_major_ae"] = df["A&E attendances Type 1"] > 0
df[["Org name", "month", "total_attendances", "over_4hrs", "pct_within_4hrs", "has_major_ae"]].head()

,Org name,month,total_attendances,over_4hrs,pct_within_4hrs,has_major_ae
0,FIRST COMMUNITY HEALTH AND CARE CIC,2025-04-01,2350,0,100.0,False
1,ASHFORD WALK-IN-CENTRE,2025-04-01,2333,49,97.9,False
2,WOKING WALK IN CENTRE,2025-04-01,2528,58,97.7,False
3,SOUTH BIRMINGHAM GP WALK IN CENTRE,2025-04-01,6628,0,100.0,False
4,PHL LYMINGTON UTC,2025-04-01,3309,42,98.7,False


In [5]:
os.makedirs("data/clean", exist_ok=True)
df.to_csv("data/clean/ae_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (2386, 27)
